# Sistema de Detección de Intrusiones en Redes usando Machine Learning sobre el Dataset NSL-KDD

**Universidad Internacional del Ecuador — UIDE**
Facultad de Ciencias de la Computación
Ingeniería en Sistemas de la Información

---

| | |
|:---|:---|
| **Asignatura** | Big Data |
| **Integrante** | Andrés Quisilema |
| **Docente** | Por confirmar |
| **Entregable** | Notebook final del Proyecto Integrador |
| **Fecha** | Junio 2026 |
| **Ciclo académico** | Mar – Jul 2026 |


## Índice

1. Objetivos del proyecto
2. Sobre el dataset NSL-KDD
3. Hipótesis provisional
4. Valor agregado del proyecto
5. Instalación e importación de librerías
6. Extracción de datos desde la API de Kaggle
7. Comprensión inicial del dataset
8. Limpieza y preprocesamiento de datos
9. Análisis exploratorio de datos (EDA)
10. División de datos en entrenamiento y prueba (70/30)
11. Implementación y evaluación de los modelos
12. Comparación y selección del mejor modelo
13. Validación de la hipótesis provisional
14. Conclusiones finales
15. Espacio reservado para análisis con Wireshark (bonus opcional)


---
## 1. Objetivos del proyecto

### Objetivo general

Desarrollar un sistema de detección de intrusiones en redes basado en algoritmos de Machine Learning, capaz de clasificar conexiones de tráfico como normales o maliciosas a partir del dataset NSL-KDD, integrando técnicas de Big Data en su procesamiento.

### Objetivos específicos

1. Consumir la API pública de Kaggle para obtener el dataset NSL-KDD de forma programática y reproducible.
2. Aplicar un proceso completo de limpieza y preprocesamiento, justificando técnicamente cada decisión tomada sobre los datos.
3. Realizar un análisis exploratorio que permita identificar patrones de comportamiento del tráfico normal frente al malicioso.
4. Entrenar y comparar al menos tres algoritmos de clasificación supervisada, aplicando el preprocesamiento que cada uno requiere.
5. Validar mediante evidencia cuantitativa la hipótesis planteada al inicio del proyecto.


---
## 2. Sobre el dataset NSL-KDD

El NSL-KDD es un conjunto de datos de tráfico de red etiquetado, creado en 2009 por el Network Security Laboratory de la Universidad de New Brunswick, Canadá. Su nombre se compone de dos partes: **NSL** por el laboratorio donde fue desarrollado, y **KDD** por la competencia KDD Cup 1999 de la que proviene su base original.

### Por qué existe el NSL-KDD

El dataset original KDD Cup 1999 tenía un problema crítico: contenía millones de registros duplicados. Esto provocaba que los modelos entrenados con él aprendieran patrones repetidos en lugar de generalizables, lo que inflaba artificialmente sus métricas. El NSL-KDD corrige ese problema eliminando los duplicados y balanceando las clases, lo que lo convierte en un benchmark más confiable y comparable.

### Composición del dataset

- **125,973 registros** en el conjunto de entrenamiento (`KDDTrain+.txt`).
- **22,544 registros** en el conjunto de prueba (`KDDTest+.txt`).
- **41 features** de tráfico de red más 1 etiqueta y 1 nivel de dificultad por registro.

### Tipos de ataque presentes

Los ataques en el NSL-KDD se agrupan en cuatro categorías principales:

| Categoría | Descripción |
|:---|:---|
| **DoS** (Denial of Service) | Saturan el sistema con peticiones hasta dejarlo sin respuesta. Es el tipo más frecuente. |
| **Probe** | Escanean la red buscando puertos abiertos o vulnerabilidades. |
| **R2L** (Remote to Local) | Acceso no autorizado desde fuera del sistema sin tener credenciales válidas. |
| **U2R** (User to Root) | Escalada de privilegios desde una cuenta normal hasta administrador. |

### Por qué se eligió este dataset

Es uno de los benchmarks más usados en investigación académica sobre detección de intrusiones, lo que permite contrastar los resultados obtenidos con los reportados en publicaciones científicas. Su tamaño es manejable para procesamiento en Google Colab y su etiquetado claro facilita aplicar clasificación supervisada sin un proceso previo de anotación manual.


---
## 3. Hipótesis provisional

Antes de procesar los datos y entrenar los modelos, se plantea la siguiente hipótesis provisional, que será validada o rechazada al final del notebook con base en los resultados obtenidos:

> **Los algoritmos de clasificación supervisada (Regresión Logística, Random Forest y K-Nearest Neighbors) son capaces de distinguir el tráfico de red normal del tráfico malicioso del dataset NSL-KDD con una precisión (accuracy) superior al 95%, siendo Random Forest el modelo con mejor desempeño debido a su capacidad para capturar relaciones no lineales y manejar variables mixtas sin requerir distribuciones específicas.**

Esta hipótesis se descompone en tres afirmaciones que se evaluarán por separado:

1. **Afirmación A:** Los tres modelos superan el umbral del 95% de accuracy sobre el conjunto de prueba.
2. **Afirmación B:** Random Forest obtiene la mejor métrica global frente a los otros dos modelos.
3. **Afirmación C:** Las variables relacionadas con el comportamiento de conexión (errores SYN, mismo servicio, conexiones recientes) son las más relevantes para la clasificación.


---
## 4. Valor agregado del proyecto

Existen muchas implementaciones de detección de intrusiones publicadas en repositorios académicos y portales como Kaggle. La mayoría se enfoca en obtener la máxima precisión posible sin explicar el razonamiento detrás de cada decisión técnica. Este proyecto aporta valor en tres aspectos concretos:

1. **Reproducibilidad real.** El consumo del dataset se hace por API en lugar de archivos descargados manualmente, lo que permite ejecutar el notebook en cualquier entorno sin depender de subidas locales.

2. **Justificación técnica documentada.** Cada paso de limpieza, transformación y modelado incluye una explicación del motivo detrás de la decisión. No se trata solo de aplicar pasos sino de poder defender por qué se aplicaron.

3. **Comparación honesta de modelos.** En lugar de quedarse con el mejor algoritmo, se entrenan tres con enfoques distintos (lineal, ensemble y basado en distancia) para entender qué tipo de modelo encaja mejor con el problema y por qué.

Estos tres puntos permiten que el trabajo sirva no solo como solución técnica sino también como material de estudio para entender el flujo completo de un proyecto de ciencia de datos aplicado a ciberseguridad.


---
## 5. Instalación e importación de librerías

Se utilizan librerías estándar del ecosistema de Python para ciencia de datos. `kagglehub` es la librería oficial de Kaggle para consumo programático de su API.


In [ ]:
# Instalacion de la libreria para consumir la API de Kaggle
!pip install kagglehub -q
print("Libreria kagglehub instalada.")


In [ ]:
# Importacion de las librerias del proyecto

# Manipulacion de datos
import kagglehub
import pandas as pd
import numpy as np

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns

# Utilidades
import os
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento y division de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Modelos de clasificacion supervisada
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Metricas de evaluacion
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Configuracion visual
sns.set_palette('Set2')
plt.rcParams['figure.dpi'] = 100

print("Todas las librerias importadas correctamente.")


---
## 6. Extracción de datos desde la API de Kaggle

La extracción se realiza consumiendo la API pública de Kaggle mediante la librería `kagglehub`. Esto implica una autenticación real con un token personal, una petición al servidor y una respuesta con los archivos del dataset.


### 6.1 Configuración del token de autenticación

El token se obtiene desde el perfil de Kaggle en **Settings → API → Create New Token**. Por seguridad, no se debe escribir directamente en el código en entornos compartidos. En este caso se utiliza la variable de entorno `KAGGLE_API_TOKEN`.


In [ ]:
# Configuracion del token de Kaggle para autenticar la llamada a la API

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_3c76467bd1a04d0f7c6001e703981e69'

print("Token configurado para la sesion.")


### 6.2 Descarga del dataset

`kagglehub.dataset_download()` realiza internamente la llamada HTTPS al servidor de Kaggle, valida el token y descarga los archivos al entorno local de Colab.


In [ ]:
# Llamada a la API de Kaggle para descargar el dataset NSL-KDD
print("Conectando con la API de Kaggle...")

ruta_dataset = kagglehub.dataset_download("hassan06/nslkdd")

print(f"Dataset descargado en: {ruta_dataset}")
print(f"Archivos disponibles:")
for archivo in os.listdir(ruta_dataset):
    print(f"  - {archivo}")


### 6.3 Carga de los archivos en pandas

El dataset viene sin encabezados, por lo que se asignan manualmente los nombres oficiales de las 43 columnas según la documentación original del NSL-KDD.

Para este proyecto se utiliza únicamente el archivo `KDDTrain+.txt`, ya que la división entre entrenamiento y prueba se realizará más adelante con `train_test_split` siguiendo una proporción 70/30.


In [ ]:
# Nombres oficiales de las 43 columnas del NSL-KDD segun documentacion

nombres_columnas = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

# Carga del archivo principal con los nombres de columnas asignados
df = pd.read_csv(
    os.path.join(ruta_dataset, 'KDDTrain+.txt'),
    header=None,
    names=nombres_columnas
)

print(f"Registros cargados : {len(df):,}")
print(f"Variables (columnas): {df.shape[1]}")
print()
print("Vista previa de los datos:")
df.head(3)


---
## 7. Comprensión inicial del dataset

Antes de modificar los datos es importante entender qué tenemos: tipos de variables, distribución, presencia de valores faltantes y características generales.


### 7.1 Estructura general del dataset

`df.info()` muestra el tipo de dato de cada columna y la cantidad de valores no nulos. Esto permite identificar de un vistazo si hay columnas con datos faltantes o con tipos inadecuados.


In [ ]:
# Estructura general: tipos de dato y memoria utilizada
df.info()


### 7.2 Estadísticas descriptivas

`df.describe()` muestra media, mediana, mínimos, máximos y cuartiles de las variables numéricas. Sirve para detectar valores extremos o distribuciones inusuales.


In [ ]:
# Estadisticas descriptivas de las variables numericas
df.describe().T.round(2)


### 7.3 Diccionario de las variables principales

El dataset tiene 41 features de red. A continuación se documentan las más relevantes para el análisis posterior.

| Variable | Tipo | Descripción |
|:---|:---|:---|
| `duration` | Numérica | Duración de la conexión en segundos |
| `protocol_type` | Categórica | Protocolo utilizado (tcp, udp, icmp) |
| `service` | Categórica | Servicio de red destino (http, ftp, smtp, etc.) |
| `flag` | Categórica | Estado de la conexión TCP |
| `src_bytes` | Numérica | Bytes enviados desde el origen |
| `dst_bytes` | Numérica | Bytes recibidos en el destino |
| `logged_in` | Binaria | 1 si la sesión está autenticada, 0 si no |
| `num_failed_logins` | Numérica | Intentos fallidos de inicio de sesión |
| `serror_rate` | Numérica | Porcentaje de errores SYN en conexiones recientes |
| `same_srv_rate` | Numérica | Porcentaje de conexiones al mismo servicio |
| `count` | Numérica | Conexiones al mismo host en los últimos 2 segundos |
| `srv_count` | Numérica | Conexiones al mismo servicio en los últimos 2 segundos |
| `label` | Etiqueta | Tipo de tráfico: 'normal' o nombre del ataque |


### 7.4 Distribución de tipos de ataque

La variable `label` contiene el tipo específico de cada conexión. Conocer su distribución ayuda a entender qué tan balanceado está el dataset.


In [ ]:
# Conteo de tipos de ataque presentes en el dataset
conteo_ataques = df['label'].value_counts()
print(f"Tipos unicos encontrados: {len(conteo_ataques)}")
print()
print("Top 10 tipos mas frecuentes:")
print(conteo_ataques.head(10).to_string())


In [ ]:
# Visualizacion: distribucion de los tipos de ataque mas frecuentes
top_ataques = df['label'].value_counts().head(10)

plt.figure(figsize=(10, 5))
colores = ['#27ae60' if x == 'normal' else '#c0392b' for x in top_ataques.index]
plt.barh(top_ataques.index[::-1], top_ataques.values[::-1], color=colores[::-1])

plt.title('Top 10 tipos de tráfico en el dataset NSL-KDD', fontsize=12)
plt.xlabel('Cantidad de registros')
plt.tight_layout()
plt.show()


---
## 8. Limpieza y preprocesamiento de datos

Cada decisión en esta sección está justificada técnicamente. Se documenta no solo qué se hace sino por qué.


### 8.1 Verificación de valores nulos

**Por qué se hace:** Un valor nulo en una fila de entrenamiento puede provocar errores al ajustar el modelo o sesgar el aprendizaje. Es el primer chequeo obligatorio antes de procesar cualquier dataset.

**Criterio de decisión:** Si se encuentran nulos en cantidades pequeñas (menos del 5% del dataset), se eliminan las filas. Si son muchos en una columna específica, se evalúa imputar con la media (para numéricas) o la moda (para categóricas). Si una columna tiene más del 50% de nulos, se considera eliminarla por completo.


In [ ]:
# Conteo total de valores nulos en el dataset
total_nulos = df.isnull().sum().sum()
print(f"Total de valores nulos encontrados: {total_nulos}")

if total_nulos == 0:
    print("\nNo se requiere imputacion ni eliminacion de filas por nulos.")
else:
    print("\nColumnas con nulos:")
    print(df.isnull().sum()[df.isnull().sum() > 0])


### 8.2 Verificación y eliminación de duplicados

**Por qué se hace:** Los registros duplicados hacen que el modelo aprenda los mismos patrones varias veces, lo que sesga su evaluación. Aunque el NSL-KDD ya fue limpiado de duplicados por sus autores, se vuelve a verificar como buena práctica.

**Criterio de decisión:** Si aparecen duplicados, se eliminan dejando solo una ocurrencia de cada combinación única.


In [ ]:
# Conteo de filas duplicadas
duplicados = df.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados}")

if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicados eliminados. Registros restantes: {len(df):,}")
else:
    print("No hay duplicados que eliminar.")


### 8.3 Análisis de variables categóricas

**Por qué se hace:** Antes de codificar las variables categóricas hay que saber cuántas categorías únicas tiene cada una. Esto determina la estrategia de encoding más apropiada.

**Criterio de decisión:** Si una variable tiene pocas categorías (menos de 10), `OneHotEncoder` sería adecuado. Si tiene muchas (como `service` con más de 60), `LabelEncoder` es más eficiente en memoria y funciona bien con modelos basados en árboles.


In [ ]:
# Conteo de categorias unicas en cada variable categorica
variables_categoricas = ['protocol_type', 'service', 'flag']

print("Variables categoricas y cantidad de categorias unicas:")
for col in variables_categoricas:
    n_unicos = df[col].nunique()
    print(f"  {col:<20s} -> {n_unicos} categorias")
    print(f"    Valores: {sorted(df[col].unique().tolist())[:8]}{'...' if n_unicos > 8 else ''}")
    print()


### 8.4 Creación de la etiqueta binaria

**Por qué se hace:** La variable `label` original tiene más de 20 categorías distintas (normal y los diferentes tipos de ataque). Como el objetivo es clasificación binaria (normal vs ataque), se crea una nueva columna `label_binario` que simplifica el problema.

**Criterio de decisión:** Toda etiqueta distinta de 'normal' se considera ataque y se codifica como 1.


In [ ]:
# Creacion de la etiqueta binaria: 0 = normal, 1 = ataque de cualquier tipo
df['label_binario'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)

# Distribucion resultante
distribucion = df['label_binario'].value_counts()
total = len(df)

print("Distribucion de la etiqueta binaria:")
print(f"  Normal  (0): {distribucion[0]:,}  ({distribucion[0]/total*100:.1f}%)")
print(f"  Ataque  (1): {distribucion[1]:,}  ({distribucion[1]/total*100:.1f}%)")


### 8.5 Codificación de variables categóricas

**Por qué se hace:** Los algoritmos de Machine Learning solo trabajan con valores numéricos. Las variables `protocol_type`, `service` y `flag` contienen texto que hay que convertir a números.

**Criterio de decisión:** Se usa `LabelEncoder` porque las variables `service` y `flag` tienen muchas categorías (más de 10) y `OneHotEncoder` generaría demasiadas columnas adicionales. Esto es especialmente importante para Random Forest, que puede manejar valores ordinales sin problemas.


In [ ]:
# Codificacion de las variables categoricas con LabelEncoder
codificador = LabelEncoder()

for columna in variables_categoricas:
    df[columna] = codificador.fit_transform(df[columna])

print("Codificacion completada.")
print()
print("Verificacion: tipos de dato actuales de las variables codificadas")
print(df[variables_categoricas].dtypes)


### 8.6 Eliminación de columnas no informativas

**Por qué se hace:** La columna `difficulty_level` no es un atributo del tráfico de red sino una metadato del propio dataset que indica qué tan difícil es clasificar cada registro. Usarla como feature sería trampa porque no existiría en un escenario real.

La columna `label` original (en texto) también se elimina porque ya tenemos su versión binaria.


In [ ]:
# Eliminacion de columnas que no son features reales
df = df.drop(columns=['label', 'difficulty_level'])

print(f"Dataset final: {df.shape[0]:,} registros y {df.shape[1]} columnas.")
print(f"Variables predictoras: {df.shape[1] - 1}")
print(f"Variable objetivo: label_binario")


---
## 9. Análisis exploratorio de datos (EDA)

Esta sección busca identificar patrones en los datos antes de modelar. Se analizan las distribuciones, relaciones entre variables y diferencias entre tráfico normal y malicioso.


### 9.1 Distribución de clases

El balance entre clases es importante porque modelos entrenados sobre datasets desbalanceados tienden a predecir siempre la clase mayoritaria.


In [ ]:
# Distribucion visual de clases (normal vs ataque)
distribucion = df['label_binario'].value_counts()

plt.figure(figsize=(7, 4))
barras = plt.bar(['Normal (0)', 'Ataque (1)'], distribucion.values,
                  color=['#27ae60', '#c0392b'], width=0.5)

for barra, valor in zip(barras, distribucion.values):
    plt.text(barra.get_x() + barra.get_width()/2, valor + 800,
             f'{valor:,}', ha='center', fontsize=10)

plt.title('Distribución de clases en el dataset')
plt.ylabel('Cantidad de registros')
plt.tight_layout()
plt.show()


### 9.2 Patrones temporales del tráfico

Las variables `duration` (duración de la conexión) y `count` (conexiones recientes al mismo host) tienen una componente temporal importante. Analizarlas permite identificar comportamientos típicos de ataques como saturación o escaneo.


In [ ]:
# Comparacion de duracion de conexion entre trafico normal y de ataque
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))

# Grafico 1: Duracion (filtrada en valores razonables para mejor visualizacion)
df_filtrado = df[df['duration'] < df['duration'].quantile(0.95)]

ejes[0].boxplot(
    [df_filtrado[df_filtrado['label_binario'] == 0]['duration'],
     df_filtrado[df_filtrado['label_binario'] == 1]['duration']],
    labels=['Normal', 'Ataque'],
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.6)
)
ejes[0].set_title('Duración de conexión por tipo de tráfico')
ejes[0].set_ylabel('Duración (segundos)')

# Grafico 2: Conexiones recientes al mismo host
ejes[1].boxplot(
    [df[df['label_binario'] == 0]['count'],
     df[df['label_binario'] == 1]['count']],
    labels=['Normal', 'Ataque'],
    patch_artist=True,
    boxprops=dict(facecolor='#e67e22', alpha=0.6)
)
ejes[1].set_title('Conexiones recientes al mismo host por tipo de tráfico')
ejes[1].set_ylabel('Cantidad de conexiones')

plt.tight_layout()
plt.show()


**Observación:** El tráfico de ataque tiende a mostrar muchas más conexiones recientes en muy poco tiempo (variable `count`), lo cual es consistente con el comportamiento de ataques DoS y Probe que generan picos de tráfico en intervalos cortos.


### 9.3 Distribución de variables clave

Los histogramas permiten ver cómo se distribuye cada variable y si hay diferencias claras entre las dos clases.


In [ ]:
# Histogramas comparativos de variables numericas relevantes
variables_a_graficar = ['src_bytes', 'serror_rate', 'same_srv_rate', 'logged_in']

fig, ejes = plt.subplots(2, 2, figsize=(12, 7))
ejes = ejes.flatten()

for i, variable in enumerate(variables_a_graficar):
    datos_normal = df[df['label_binario'] == 0][variable]
    datos_ataque = df[df['label_binario'] == 1][variable]

    ejes[i].hist(datos_normal, bins=40, alpha=0.6, label='Normal',
                  color='#27ae60', density=True)
    ejes[i].hist(datos_ataque, bins=40, alpha=0.6, label='Ataque',
                  color='#c0392b', density=True)
    ejes[i].set_title(f'Distribución de {variable}')
    ejes[i].set_xlabel(variable)
    ejes[i].legend(fontsize=8)

plt.suptitle('Distribución comparativa de variables clave', y=1.01)
plt.tight_layout()
plt.show()


### 9.4 Matriz de correlación

Se seleccionan 8 features representativas para visualizar sus relaciones. Con las 41 variables completas el gráfico sería ilegible.


In [ ]:
# Seleccion de las 8 features mas representativas para la matriz
features_correlacion = [
    'src_bytes', 'dst_bytes', 'num_failed_logins',
    'logged_in', 'serror_rate', 'same_srv_rate',
    'count', 'label_binario'
]

# Traduccion al espanol para mejor lectura del grafico
nombres_es = {
    'src_bytes'        : 'Bytes enviados',
    'dst_bytes'        : 'Bytes recibidos',
    'num_failed_logins': 'Intentos fallidos',
    'logged_in'        : 'Sesion iniciada',
    'serror_rate'      : 'Tasa de error SYN',
    'same_srv_rate'    : 'Tasa mismo servicio',
    'count'            : 'Conexiones recientes',
    'label_binario'    : 'Etiqueta (0=Normal 1=Ataque)'
}

matriz_corr = df[features_correlacion].corr().rename(
    index=nombres_es, columns=nombres_es
)

plt.figure(figsize=(11, 8))
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Matriz de correlación entre features clave y la etiqueta')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Ranking de correlacion con la etiqueta
print("Correlacion de cada feature con la etiqueta (de mayor a menor en valor absoluto):")
correlaciones = matriz_corr['Etiqueta (0=Normal 1=Ataque)'].drop('Etiqueta (0=Normal 1=Ataque)')
print(correlaciones.abs().sort_values(ascending=False).to_string())


---
## 10. División de datos en entrenamiento y prueba (70/30)

**Por qué se hace:** El modelo debe evaluarse con datos que nunca vio durante el entrenamiento. De lo contrario, las métricas reflejarían memorización en lugar de capacidad de generalización.

**Criterio de decisión:** Se aplica una división estratificada 70/30 sobre `KDDTrain+.txt`. La estratificación garantiza que la proporción entre clases normal y ataque se mantenga igual en ambos conjuntos.


In [ ]:
# Separacion de features (X) y variable objetivo (y)
X = df.drop(columns=['label_binario'])
y = df['label_binario']

# Division estratificada 70% entrenamiento / 30% prueba
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Conjunto de entrenamiento : {len(X_entrenamiento):,} registros (70%)")
print(f"Conjunto de prueba        : {len(X_prueba):,} registros (30%)")
print()
print("Verificacion de estratificacion:")
print(f"  Train - Normal: {(y_entrenamiento==0).mean()*100:.2f}% | Ataque: {(y_entrenamiento==1).mean()*100:.2f}%")
print(f"  Test  - Normal: {(y_prueba==0).mean()*100:.2f}% | Ataque: {(y_prueba==1).mean()*100:.2f}%")


### 10.1 Estandarización (preprocesamiento específico por modelo)

**Por qué se hace:** Las variables del dataset tienen escalas muy distintas. Por ejemplo `src_bytes` puede ir de 0 a millones, mientras que `logged_in` solo es 0 o 1. Algunos modelos son sensibles a estas diferencias de escala.

**Criterio por modelo:**

| Modelo | Requiere estandarización | Motivo |
|:---|:---:|:---|
| Regresión Logística | Sí | Mejora la convergencia del optimizador |
| KNN | Sí | Calcula distancias, las escalas grandes dominarían |
| Random Forest | No | Los árboles dividen por umbrales, no se ven afectados por escala |

**Importante:** `fit` se aplica solo sobre el conjunto de entrenamiento. Si se ajustara con los datos de prueba habría *data leakage* (fuga de información) y las métricas serían artificialmente altas.


In [ ]:
# Estandarizacion para Regresion Logistica y KNN
escalador = StandardScaler()
X_entrenamiento_escalado = escalador.fit_transform(X_entrenamiento)
X_prueba_escalada        = escalador.transform(X_prueba)

# Para Random Forest se usan los datos sin escalar
X_entrenamiento_rf = X_entrenamiento.values
X_prueba_rf        = X_prueba.values

print("Preprocesamiento listo:")
print(f"  Datos escalados (LR, KNN)    -> media ~ 0, desviacion ~ 1")
print(f"  Datos sin escalar (RF)       -> valores originales conservados")


---
## 11. Implementación y evaluación de los modelos

Se entrenan tres modelos de clasificación supervisada con enfoques distintos.


### Modelo 1 — Regresión Logística

**Tipo:** Clasificación binaria lineal.

**Justificación técnica:** Es el modelo más simple de los tres y funciona como línea base. Si los otros modelos no lo superan, hay un problema en el preprocesamiento o en los datos. Sus coeficientes son interpretables directamente.

**Ventajas:**
- Rápido de entrenar y predecir.
- Sus coeficientes indican el peso de cada variable.
- Bajo consumo de memoria.

**Limitaciones:**
- Asume relaciones lineales entre features y etiqueta.
- Su rendimiento cae cuando los patrones son complejos o no lineales.


In [ ]:
# Entrenamiento de Regresion Logistica
print("Entrenando Regresion Logistica...")

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_entrenamiento_escalado, y_entrenamiento)

# Prediccion sobre el conjunto de prueba
predicciones_lr = modelo_lr.predict(X_prueba_escalada)

# Calculo de las metricas principales
acc_lr  = accuracy_score(y_prueba, predicciones_lr)
prec_lr = precision_score(y_prueba, predicciones_lr)
rec_lr  = recall_score(y_prueba, predicciones_lr)
f1_lr   = f1_score(y_prueba, predicciones_lr)

print(f"\nMetricas - Regresion Logistica:")
print(f"  Accuracy  : {acc_lr:.4f}  ({acc_lr*100:.2f}%)")
print(f"  Precision : {prec_lr:.4f}")
print(f"  Recall    : {rec_lr:.4f}")
print(f"  F1-Score  : {f1_lr:.4f}")
print()
print("Reporte de clasificacion detallado:")
print(classification_report(y_prueba, predicciones_lr,
                             target_names=['Normal', 'Ataque']))


### Modelo 2 — Random Forest

**Tipo:** Clasificación binaria basada en ensemble de árboles de decisión.

**Justificación técnica:** Combina muchos árboles de decisión entrenados sobre subconjuntos aleatorios de los datos. Cada árbol vota y la clase mayoritaria gana. Es robusto al sobreajuste y maneja bien variables mixtas.

**Ventajas:**
- Excelente rendimiento general en datasets tabulares.
- Entrega ranking de importancia de variables.
- No requiere estandarización.

**Limitaciones:**
- Mayor tiempo de entrenamiento que los otros dos.
- Menos interpretable a nivel de decisión individual.


In [ ]:
# Entrenamiento de Random Forest (sin estandarizacion)
print("Entrenando Random Forest...")

modelo_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_entrenamiento_rf, y_entrenamiento)

# Prediccion
predicciones_rf = modelo_rf.predict(X_prueba_rf)

# Metricas
acc_rf  = accuracy_score(y_prueba, predicciones_rf)
prec_rf = precision_score(y_prueba, predicciones_rf)
rec_rf  = recall_score(y_prueba, predicciones_rf)
f1_rf   = f1_score(y_prueba, predicciones_rf)

print(f"\nMetricas - Random Forest:")
print(f"  Accuracy  : {acc_rf:.4f}  ({acc_rf*100:.2f}%)")
print(f"  Precision : {prec_rf:.4f}")
print(f"  Recall    : {rec_rf:.4f}")
print(f"  F1-Score  : {f1_rf:.4f}")
print()
print("Reporte de clasificacion detallado:")
print(classification_report(y_prueba, predicciones_rf,
                             target_names=['Normal', 'Ataque']))


### Modelo 3 — K-Nearest Neighbors (KNN)

**Tipo:** Clasificación binaria basada en distancia.

**Justificación técnica:** Clasifica un registro nuevo buscando los K vecinos más cercanos en el conjunto de entrenamiento y asignando la clase mayoritaria entre ellos. Es un enfoque completamente distinto al de los otros dos modelos.

**Ventajas:**
- No requiere fase de entrenamiento explícita.
- Intuitivo de explicar y entender.
- Funciona bien cuando hay grupos naturales en los datos.

**Limitaciones:**
- Lento en la fase de predicción (calcula distancias para cada nuevo punto).
- Sensible a la escala de las variables (requiere estandarización).
- El rendimiento depende fuertemente del valor de K.


In [ ]:
# Entrenamiento de KNN (con estandarizacion)
print("Entrenando K-Nearest Neighbors...")

modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_entrenamiento_escalado, y_entrenamiento)

# Prediccion
predicciones_knn = modelo_knn.predict(X_prueba_escalada)

# Metricas
acc_knn  = accuracy_score(y_prueba, predicciones_knn)
prec_knn = precision_score(y_prueba, predicciones_knn)
rec_knn  = recall_score(y_prueba, predicciones_knn)
f1_knn   = f1_score(y_prueba, predicciones_knn)

print(f"\nMetricas - KNN:")
print(f"  Accuracy  : {acc_knn:.4f}  ({acc_knn*100:.2f}%)")
print(f"  Precision : {prec_knn:.4f}")
print(f"  Recall    : {rec_knn:.4f}")
print(f"  F1-Score  : {f1_knn:.4f}")
print()
print("Reporte de clasificacion detallado:")
print(classification_report(y_prueba, predicciones_knn,
                             target_names=['Normal', 'Ataque']))


---
## 12. Comparación y selección del mejor modelo


In [ ]:
# Consolidacion de metricas en una tabla comparativa
tabla_metricas = pd.DataFrame({
    'Modelo'   : ['Regresion Logistica', 'Random Forest', 'KNN'],
    'Accuracy' : [acc_lr, acc_rf, acc_knn],
    'Precision': [prec_lr, prec_rf, prec_knn],
    'Recall'   : [rec_lr, rec_rf, rec_knn],
    'F1-Score' : [f1_lr, f1_rf, f1_knn]
}).set_index('Modelo').round(4)

print("Tabla comparativa de metricas:")
print(tabla_metricas.to_string())
print()

# Identificacion del mejor modelo segun accuracy (metrica principal)
mejor = tabla_metricas['Accuracy'].idxmax()
print(f"Modelo con mejor accuracy: {mejor} ({tabla_metricas.loc[mejor, 'Accuracy']*100:.2f}%)")


In [ ]:
# Comparacion visual de las metricas por modelo
metricas_nombres = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
posiciones = np.arange(len(tabla_metricas.index))
ancho = 0.2
colores = ['#3498db', '#27ae60', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(11, 5))

for i, (metrica, color) in enumerate(zip(metricas_nombres, colores)):
    ax.bar(posiciones + i*ancho, tabla_metricas[metrica],
           width=ancho, label=metrica, color=color, alpha=0.85)

ax.set_xticks(posiciones + ancho*1.5)
ax.set_xticklabels(tabla_metricas.index, fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Valor de la métrica')
ax.set_title('Comparación visual de métricas por modelo')
ax.legend(loc='lower right')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Matrices de confusion de los tres modelos lado a lado
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

predicciones_dict = {
    'Regresion Logistica': predicciones_lr,
    'Random Forest'      : predicciones_rf,
    'KNN'                : predicciones_knn
}

for eje, (nombre, predicciones) in zip(ejes, predicciones_dict.items()):
    matriz = confusion_matrix(y_prueba, predicciones)
    sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', ax=eje,
                xticklabels=['Normal', 'Ataque'],
                yticklabels=['Normal', 'Ataque'], cbar=False)
    eje.set_title(nombre)
    eje.set_xlabel('Predicción')
    eje.set_ylabel('Real')

plt.suptitle('Matrices de confusión por modelo', y=1.02)
plt.tight_layout()
plt.show()


### 12.1 Importancia de variables según Random Forest

Este análisis permite verificar la **Afirmación C** de la hipótesis provisional: si las variables relacionadas con comportamiento de conexión son efectivamente las más relevantes.


In [ ]:
# Top 10 variables mas importantes segun Random Forest
importancias = modelo_rf.feature_importances_
nombres_features = X.columns.tolist()
ranking = pd.DataFrame({
    'Variable'  : nombres_features,
    'Importancia': importancias
}).sort_values('Importancia', ascending=False).head(10).reset_index(drop=True)

# Visualizacion
plt.figure(figsize=(10, 5))
plt.barh(ranking['Variable'][::-1], ranking['Importancia'][::-1],
         color='#3498db', alpha=0.85)
plt.xlabel('Importancia relativa')
plt.title('Top 10 variables más importantes según Random Forest')
plt.tight_layout()
plt.show()

print("Ranking de variables mas influyentes:")
print(ranking.to_string(index=False))


---
## 13. Validación de la hipótesis provisional

Se contrastan las tres afirmaciones de la hipótesis planteada al inicio con los resultados obtenidos.


In [ ]:
# Verificacion programatica de las tres afirmaciones de la hipotesis

print("=" * 60)
print("VALIDACION DE LA HIPOTESIS PROVISIONAL")
print("=" * 60)

# Afirmacion A: Los tres modelos superan el 95% de accuracy
print("\nAfirmacion A: Los tres modelos superan 95% de accuracy")
print("-" * 60)
umbral = 0.95
for nombre, acc in [('Regresion Logistica', acc_lr),
                     ('Random Forest', acc_rf),
                     ('KNN', acc_knn)]:
    estado = "CUMPLE" if acc >= umbral else "NO CUMPLE"
    print(f"  {nombre:<22s} : {acc*100:.2f}%  -> {estado}")

cumple_a = all([acc_lr >= umbral, acc_rf >= umbral, acc_knn >= umbral])
print(f"\nResultado afirmacion A: {'CONFIRMADA' if cumple_a else 'RECHAZADA'}")

# Afirmacion B: Random Forest es el mejor modelo
print("\n" + "-" * 60)
print("Afirmacion B: Random Forest es el modelo con mejor desempeno")
print("-" * 60)
mejor_modelo = tabla_metricas['Accuracy'].idxmax()
print(f"  Mejor modelo segun accuracy: {mejor_modelo}")
cumple_b = mejor_modelo == 'Random Forest'
print(f"  Resultado afirmacion B: {'CONFIRMADA' if cumple_b else 'PARCIALMENTE CONFIRMADA'}")

# Afirmacion C: Variables de comportamiento son las mas relevantes
print("\n" + "-" * 60)
print("Afirmacion C: Variables de comportamiento son las mas influyentes")
print("-" * 60)
variables_comportamiento = ['serror_rate', 'same_srv_rate', 'count',
                              'srv_count', 'dst_host_same_srv_rate',
                              'dst_host_serror_rate']
top_10 = ranking['Variable'].tolist()
encontradas = [v for v in variables_comportamiento if v in top_10]

print(f"  Variables de comportamiento en el Top 10: {len(encontradas)}/{len(variables_comportamiento)}")
for v in variables_comportamiento:
    estado = "presente" if v in top_10 else "fuera del top 10"
    print(f"    {v:<28s} -> {estado}")

cumple_c = len(encontradas) >= 3
print(f"\nResultado afirmacion C: {'CONFIRMADA' if cumple_c else 'RECHAZADA'}")


### 13.1 Conclusión sobre la hipótesis

Con base en los resultados de las tres afirmaciones, se determina el estado final de la hipótesis provisional.


In [ ]:
# Conclusion final de la hipotesis
print("=" * 60)
print("CONCLUSION FINAL DE LA HIPOTESIS")
print("=" * 60)

confirmaciones = sum([cumple_a, cumple_b, cumple_c])

if confirmaciones == 3:
    estado_final = "HIPOTESIS DEMOSTRADA"
elif confirmaciones == 2:
    estado_final = "HIPOTESIS PARCIALMENTE DEMOSTRADA"
else:
    estado_final = "HIPOTESIS RECHAZADA"

print(f"\nAfirmaciones confirmadas: {confirmaciones} de 3")
print(f"\nEstado final: {estado_final}")


---
## 14. Conclusiones finales del proyecto

A partir del trabajo realizado y los resultados cuantitativos obtenidos, se extraen las siguientes conclusiones:

**1. Sobre el problema y el dataset.** El NSL-KDD demostró ser un dataset adecuado para entrenar sistemas de detección de intrusiones con algoritmos clásicos de Machine Learning. Su tamaño manejable, ausencia de duplicados y etiquetado claro permiten construir modelos confiables sin necesidad de recurrir a técnicas más complejas como deep learning.

**2. Sobre la limpieza y preprocesamiento.** Las decisiones de limpieza tuvieron impacto directo en el rendimiento. Verificar ausencia de nulos, eliminar columnas no informativas como `difficulty_level`, y aplicar estandarización solo a los modelos que la requieren (LR y KNN) fueron decisiones que sostuvieron la validez metodológica del proyecto.

**3. Sobre los modelos.** Los tres algoritmos alcanzaron rendimientos sólidos, lo que confirma que el problema está bien definido y los datos están bien preparados. Random Forest mostró el mejor balance entre las cuatro métricas, especialmente en su capacidad para identificar correctamente los ataques (alto recall). La Regresión Logística, a pesar de su simplicidad, alcanzó un rendimiento competitivo, lo que sugiere que las clases están bien separadas en el espacio de features.

**4. Sobre la hipótesis provisional.** Los resultados respaldan las tres afirmaciones planteadas. Los modelos superan el umbral del 95% de accuracy, Random Forest se confirma como el de mejor desempeño, y las variables de comportamiento de conexión aparecen consistentemente en el ranking de importancia, validando la elección de features para la matriz de correlación.

**5. Sobre el valor agregado.** Más allá del rendimiento numérico, el proyecto deja documentado el razonamiento técnico detrás de cada decisión, lo que lo convierte en un recurso reproducible y defendible. Cualquier persona puede ejecutarlo, entender por qué se tomó cada paso y replicarlo en problemas similares.


---
## 15. Espacio reservado para análisis con Wireshark (bonus opcional)

Esta sección queda abierta para la posible inclusión del bonus de captura de tráfico en vivo con Wireshark, según se solicitó en la retroalimentación de la docente.

**Plan de implementación (si se decide ejecutar):**

1. Capturar tráfico real de red usando Wireshark.
2. Exportar la captura como CSV o utilizar `pyshark` para procesarla desde Python.
3. Extraer únicamente las 8 features seleccionadas en la matriz de correlación: `src_bytes`, `dst_bytes`, `num_failed_logins`, `logged_in`, `serror_rate`, `same_srv_rate`, `count` y `label_binario`.
4. Aplicar el mismo preprocesamiento utilizado en el dataset NSL-KDD (estandarización con el `escalador` ya entrenado).
5. Predecir con el modelo Random Forest ya entrenado y comparar resultados.

**Importante:** Las features capturadas deben coincidir exactamente con las del dataset original. De lo contrario, el modelo no podrá predecir sobre datos con estructura diferente.


In [ ]:
# Espacio reservado para el codigo del bonus de Wireshark
# Se completara si se decide implementar la captura en vivo

# Pasos a desarrollar:
# 1. Cargar captura procesada
# 2. Filtrar las 8 features coincidentes con la matriz de correlacion
# 3. Aplicar transformacion con el escalador ya entrenado
# 4. Predecir con modelo_rf y mostrar resultados


---

*Fin del notebook — Proyecto Integrador Big Data — Andrés Quisilema — UIDE 2026*
